# Case 5 - Raven Elbow Discharge Ensemble Forecast Comparison

## Learning goals of this module
- Learn how to do a basic analysis, comparing observed precipitation with reanalysis precipitation from RDPA and HRDPA
- Learn how to run a simple example to access and compare observed and reanalysis precipitation.
- Learn how to calculate and visualize simple error metrics

## Assumptions
- We assume you are familiar with the concept of and _ensemble_ forecast.
- We assume you are familiar with the concept of a hydrological model. 

## Run imports and set-up logging

In [1]:
from pathlib import Path
from veriflow import run_pipeline
from veriflow.constants import VERSION_FULL
import logging
from dotenv import load_dotenv
import warnings
import sys

# Load plotting functions
sys.path.append(str(Path("..").resolve()))
from verification_plots import *

# Reload automatically
%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Load the environment variables
load_dotenv(dotenv_path="tutorial.env", override=True)

base_config = Path("config")
base_config.exists()

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
logging.info(f"Running Veriflow version {VERSION_FULL}")

2026-06-13 11:51:03,095 - INFO - Running Veriflow version 0.1.0+c1c26d8007b29754136821ac83fc747e40506b9b.dirty


## Inspecting the _veriflow_ pipeline configuration
1. Open the [config file](./config/Raven_Elbow_Discharge_Ensemble.yaml) for the pipeline
2. Inspect each of the sections to gain an understanding of what this configuration is about.

## Running the _veriflow_ pipeline

In [2]:
config_file_path = base_config / "Raven_Elbow_Discharge_Ensemble.yaml"
ods = run_pipeline(config=(config_file_path, "yaml"))

2026-06-13 11:51:03,166 - INFO - Successfully initialized the configuration. 
	 verification_period_start = 2025-05-20 00:00:00 
	 verification_period_end = 2026-06-01 00:00:00
2026-06-13 11:51:03,174 - INFO - Start getting data from FewsWebservice.
2026-06-13 11:51:03,650 - INFO - Download successful from URL: https://veriflow-open.fews.deltares.nl/FewsWebServices/rest/fewspiservice/v1/timeseries?locationIds=05BJ010&parameterIds=QR.obs&moduleInstanceIds=ImportWSC&startTime=2025-05-21T00%3A00%3A00Z&endTime=2026-06-07T00%3A00%3A00Z&timeSeriesType=EXTERNAL_HISTORICAL&documentFormat=PI_NETCDF
2026-06-13 11:51:05,276 - INFO - Successfully got data from FewsWebservice.
2026-06-13 11:51:05,277 - INFO - Start getting data from FewsWebservice.
2026-06-13 11:51:05,883 - INFO - Download successful from URL: https://veriflow-open.fews.deltares.nl/FewsWebServices/rest/fewspiservice/v1/timeseries?locationIds=05BJ010&parameterIds=QR.sim&moduleInstanceIds=ElbowRavenGEPSForecast&startForecastTime=2025

In [3]:
ods

OutputDataset with 3 verification pairs: [VerificationPair(id='QR_GEPS', obs='observed', sim='simulated_GEPS', variable='discharge'), VerificationPair(id='QR_GEFS', obs='observed', sim='simulated_GEFS', variable='discharge'), VerificationPair(id='QR_IFS', obs='observed', sim='simulated_IFS', variable='discharge')]

## Evaluating the results in the _veriflow_ `OutputDataset`

Verification metrics and results can contain a level of abstraction. Although these abstractions can reveal important information about forecast quality, a basic "eyeball verification" is often the best and intuitive way to start your verification exercise. You'll likely find strengths and weaknesses in your forecasts early on, without directly diving into levels of abstraction. In addition, a solid visual inspection may help you later on in understanding or explaining the more abstract results.

### 1 - Visual inspection of observed and forecast data
A good starting point for "eyeball" verification is simple: just looking at your observations and forecasts in a visual way. Use the interactive elements in the plots below to zoom, pan and compare the results of our 3 NWP products.

In [8]:
stations = ods.get(ods.verification_pairs[0]).coords["station"].values
lead_times = ods.get(ods.verification_pairs[0]).coords["lead_time"].values

In [9]:
from verification_plots import forecast_timeseries_plot
print(ods)
forecast_timeseries_plot(ods, show_members=True)


OutputDataset with 3 verification pairs: [VerificationPair(id='QR_GEPS', obs='observed', sim='simulated_GEPS', variable='discharge'), VerificationPair(id='QR_GEFS', obs='observed', sim='simulated_GEFS', variable='discharge'), VerificationPair(id='QR_IFS', obs='observed', sim='simulated_IFS', variable='discharge')]


### 2 - Visual inspection with a scatter plot per lead time
Another great tool for "eyeball" verification is the scatter plot. The scatter plot is relatively easy to understand, but is slightly more abstract than the visualization above. 

For all forecasts in our `OutputDataset`, we collect all realizations of the ensemble at a specific lead time. You can use the  `lead_times` variable  (a list of `np.timedelta64` instances) to slice the data at a specific lead time. For that given slice, we can make the scatter plot.


In [10]:
scatter_plot(ods, lead_time=lead_times[1])

### 3 - Looking into the Continuous Ranked Probability Score

Next, we'll look into the results of the continuous ranked probability score. Before you continue to the next section, we'll look into the documentation and do a tutorial on the CRPS for ensemble forecasts. The _veriflow_ package relies on _scores_ (developed by the Bureau of Meteorology, Australia) for computation of various scores. 

1. Read the [API documentation](https://scores.readthedocs.io/en/stable/api.html#scores.probability.crps_for_ensemble) the __crps_for_ensemble__ function, which is used under the hood in _veriflow_. 
2. Run through the [scores tutorial](https://scores.readthedocs.io/en/stable/tutorials/CRPS_for_Ensembles.html) on the __crps_for_ensemble__ function. You can run it in Binder (link on top of the page), or view the static view.


- Q1: what attribute(s) of forecast quality can be measured by the CRPS?

<details>
<summary>Show suggested answers for Q1</summary>
The Continuous Ranked Probability Score (CRPS) is primarily a measure of overall forecast accuracy (quality) for probabilistic forecasts.

The important nuance is that CRPS is a proper scoring rule, meaning it evaluates the entire forecast distribution against the observation. As a result, the score reflects a combination of forecast attributes rather than isolating one attribute.

For probabilistic forecasts, CRPS can be decomposed into components analogous to the Brier Score decomposition:

CRPS=Reliability−Resolution+Uncertainty.

This decomposition shows that:
- Better reliability lowers CRPS.
- Better resolution lowers CRPS.
- Higher inherent uncertainty increases CRPS.

Therefore, CRPS itself is not a measure of reliability or resolution; it is an aggregate measure of forecast accuracy whose value depends on those attributes.

</details>

- Q2: what is the best possible CRPS score?

<details>
<summary>Show suggested answers for Q2</summary>
The best possible outcome of CRPS is 0. In this case the forecast has maximum sharpness: all ensemble members exactly predict the observed outcome.
</details>


- Q3: computing the CRPS over just one realization (i.e. a deterministic forecast) yields the exact same result as computing the ... for a deterministic score?

<details>
<summary>Show suggested answers for Q3</summary>
The absolute error. The CRPS is a probabilistic generalization of the absolute error. When taking the mean of the CRPS over all forecasts, it is equal to the mean absolute error when the number of realizations is 1.
</details>

In [11]:
from verification_plots import crps_plot, scatter_plot, rank_histogram_plot

crps_plot(ods)


### 3 - Looking into the Rank Histogram

Next, we'll look into the results of the continuous ranked probability score. Before you continue to the next section, we'll look into the documentation and do a tutorial on the CRPS for ensemble forecasts. The _veriflow_ package relies on _scores_ (developed by the Bureau of Meteorology, Australia) for computation of various scores. 

1. Read the [API documentation](https://scores.readthedocs.io/en/stable/api.html#scores.probability.crps_for_ensemble) the __crps_for_ensemble__ function, which is used under the hood in _veriflow_. 
2. Run through the [scores tutorial](https://scores.readthedocs.io/en/stable/tutorials/CRPS_for_Ensembles.html) on the __crps_for_ensemble__ function. You can run it in Binder (link on top of the page), or view the static view.

- Q1: what attribute(s) of forecast quality can be measured by the Rank Histogram?

<details>
<summary>Show suggested answers for Q1</summary>
The rank histogram primarily measures reliability, although bias in forecasts will show up in the rank histogram as well.
</details>

- Q2: what does a perfect Rank Histogram look like?

<details>
<summary>Show suggested answers for Q2</summary>
The rank histogram shows a uniform (flat) distribution, indicating forecast and observed frequencies are equal.
</details>

- Q3: does a perfectly uniform/flat rank histogram always correspond to a good forecast? Can you come up with a hypothetical forecast that has a perfect rank histogram, but is sill bad?

<details>
<summary>Show suggested answers for Q3</summary>
No, although a perfect rank histogram indicates statistical reliability, the forecast may still have poor resolution (the ability to discriminate between situations that have different observed outcomes).

Imagine an ensemble forecast that ignores current conditions and simply samples from the historical distribution of streamflow for that day of year.

For example, every day in June the ensemble consists of random draws from the last 30 years of June observations.

This forecast can also produce a nearly uniform rank histogram because the observations come from the same climatological distribution. However, it has zero ability to distinguish today's conditions from any other June day.

</details>


In [12]:
from verification_plots import rank_histogram_plot

rank_histogram_plot(ods, station=stations[0], lead_time=lead_times[4])
